### Harness工程
> Harness工程更关注整体架构，这跟我最近所思考的术与道有些相似。回顾过往，我一直在追求某种确定性的解决办法或者方案，就像解题一样，但实际上不是的，最主要的是思考和想法，正如算法题和代码一样，代码应该跟随逻辑，而不是为了解决一道题目，逻辑对，follow就完事。因此，学习Harnees工程辅助核心项目架构思路，再进行细节性的代码补充，才应该是正常的工程逻辑。抓大放下，分解问题，逐步实施，这才是正确的生活、工作方法。

本项目代码库：https://github.com/shareAI-lab/learn-claude-code/blob/main/README-zh.md

2026年4月23日，正式开启项目流程，以此作为简历项目支撑之一。

---

#### Harness逻辑
Agent工程本质是单个的Agent_Loop加上思考、工具调用，最后得到相关结果。其中，最重要的模式就是agent_loop:


In [3]:

def agent_loop(messages, llm):
    while True:
        response = llm.invoke(messages)
        messages.append({"role" : "assistance",
                        "content": response.contents})
        
        if response.stop_reason != "tool_use":
            return

        results = []

        for block in response.content:
            if block.type == "tool_use":
                output = TOOL_HANDLERS[block.name](**block.input)
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": output,
                })

        messages.append({"role": "user", "content": results})

### S01 Agent循环
> "One loop & Bash is all you need" -- 一个工具 + 一个循环 = 一个 Agent。

> Harness 层: 循环 -- 模型与真实世界的第一道连接。
问题：
语言模型能推理代码, 但碰不到真实世界 -- 不能读文件、跑测试、看报错。没有循环, 每次工具调用你都得手动把结果粘回去。你自己就是那个循环。
解决方案：
```bash
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> |  Tool   |
| prompt |      |       |      | execute |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                    (loop until stop_reason != "tool_use")

In [5]:
def agent_loop(query):
    messages = [{"role": "user", "content": query}]
    while True:
        response = client.messages.create(
            model=MODEL, system=SYSTEM, messages=messages,
            tools=TOOLS, max_tokens=8000,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return

        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = run_bash(block.input["command"])
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": output,
                })
        messages.append({"role": "user", "content": results})

### S02 tooluse

> "加一个工具, 只加一个 handler" -- 循环不用动, 新工具注册进 dispatch map 就行。

> Harness 层: 工具分发 -- 扩展模型能触达的边界。

问题:
只有 bash 时, 所有操作都走 shell。cat 截断不可预测, sed 遇到特殊字符就崩, 每次 bash 调用都是不受约束的安全面。专用工具 (read_file, write_file) 可以在工具层面做路径沙箱。

关键洞察: 加工具不需要改循环。

解决方案:
```bash
+--------+      +-------+      +------------------+
|  User  | ---> |  LLM  | ---> | Tool Dispatch    |
| prompt |      |       |      | {                |
+--------+      +---+---+      |   bash: run_bash |
                    ^           |   read: run_read |
                    |           |   write: run_wr  |
                    +-----------+   edit: run_edit |
                    tool_result | }                |
                                +------------------+

The dispatch map is a dict: {tool_name: handler_function}.
One lookup replaces any if/elif chain.
```

In [25]:
# 每个工具有一个处理函数。路径沙箱防止逃逸工作区。
def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

def run_read(path: str, limit: int = None) -> str:
    text = safe_path(path).read_text()
    lines = text.splitlines()
    if limit and limit < len(lines):
        lines = lines[:limit]
    return "\n".join(lines)[:50000]

# dispatch map 将工具名映射到处理函数。
TOOL_HANDLERS= {
    "bash":       lambda **kw: run_bash(kw["command"]),
    "read_file":  lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":  lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
}

# 循环中按名称查找处理函数。循环体本身与 s01 完全一致。
for block in response.content:
    if block.type == "tool_use":
        handler = TOOL_HANDLERS.get(block.name)
        output = handler(**block.input) if handler \
            else f"Unknown tool: {block.name}"
        results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": output,
        })

In [26]:
# 以上做法本质也是在做注册工具，需要调用时直接选择

### S03 ToDoWrite
> "没有计划的 agent 走哪算哪" -- 先列步骤再动手, 完成率翻倍。

>Harness 层: 规划 -- 让模型不偏航, 但不替它画航线。

问题:
多步任务中, 模型会丢失进度 -- 重复做过的事、跳步、跑偏。对话越长越严重: 工具结果不断填满上下文, 系统提示的影响力逐渐被稀释。一个 10 步重构可能做完 1-3 步就开始即兴发挥, 因为 4-10 步已经被挤出注意力了。

解决方案:
```bash
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> | Tools   |
| prompt |      |       |      | + todo  |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                          |
              +-----------+-----------+
              | TodoManager state     |
              | [ ] task A            |
              | [>] task B  <- doing  |
              | [x] task C            |
              +-----------------------+
                          |
              if rounds_since_todo >= 3:
                inject <reminder> into tool_result

In [ ]:
# TodoManager 存储带状态的项目。同一时间只允许一个 in_progress。
class TodoManager:
    def update(self, items: list) -> str:
        validated, in_progress_count = [], 0
        for item in items:
            status = item.get("status", "pending")
            if status == "in_progress":
                in_progress_count += 1
            validated.append({"id": item["id"], "text": item["text"],
                              "status": status})
        if in_progress_count > 1:
            raise ValueError("Only one task can be in_progress")
        self.items = validated
        return self.render()
    
# todo 工具和其他工具一样加入 dispatch map。
TOOL_HANDLERS = {
    # ...base tools...
    "todo": lambda **kw: TODO.update(kw["items"]),
}

# nag reminder: 模型连续 3 轮以上不调用 todo 时注入提醒。
if rounds_since_todo >= 3 and messages:
    last = messages[-1]
    if last["role"] == "user" and isinstance(last.get("content"), list):
        last["content"].insert(0, {
            "type": "text",
            "text": "<reminder>Update your todos.</reminder>",
        })

In [4]:
import mysql.connector
db = mysql.connector.connect(
    host = '127.0.0.1',
    user = 'root',
    password = 'Root@123456',
    database = 'test_db'
)
print("数据库连接成功")
cursor = db.cursor()

数据库连接成功


In [14]:
sql = "INSERT INTO test_table (name, value) VALUES (%s, %s)"
values = ("py脚本", "来自宿主机")
cursor.execute(sql, values)
db.commit()